In [2]:
2+2

4

In [1]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
# import torch
import seaborn as sns
import matplotlib.pyplot as plt

import ast
import re


In [3]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

In [ ]:
result = pd.read_parquet("results/results_myvideos_174/logits_output_meta.parquet")
print(result.shape)
del result['top5_classes']
result.head(4)

(150, 12)


,video_id,cam,session,template,label,top1_pred,top1_class,top1_conf,top5_preds,correct_top1,correct_top5
0,260922-134214,cam0,plug_1,Plugging [something] into [something],49,49,Plugging [something] into [something],0.71,"[49, 106, 1, 145, 5]",True,True
1,260922-134214,cam1,plug_1,Plugging [something] into [something],49,49,Plugging [something] into [something],0.77,"[49, 106, 1, 145, 5]",True,True
2,260922-134214,cam2,plug_1,Plugging [something] into [something],49,49,Plugging [something] into [something],0.89,"[49, 1, 106, 96, 50]",True,True
3,260922-134301,cam0,plug_1,Plugging [something] into [something],49,49,Plugging [something] into [something],0.75,"[49, 106, 1, 5, 145]",True,True


In [ ]:
d = np.load("results/results_myvideos_174/logits_output.npz", allow_pickle=True)
print(d["logits"].shape)   # (150, 10)
d

(150, 174)


NpzFile 'results/results_myvideos_4/logits_output.npz' with keys: logits, probs, labels, video_ids, cams...

# main result

In [7]:
from sklearn.metrics import balanced_accuracy_score
    
n = len(result)
top1_acc  = result["correct_top1"].mean()
top5_acc  = result["correct_top5"].mean()
ba_top1   = balanced_accuracy_score(result["label"], result["top1_pred"])

# Top-5 balanced: credit correct if label is anywhere in top-5
top5_pred_adj = [
    row["label"] if row["correct_top5"] else row["top5_preds"][0]
    for _, row in result.iterrows()
]
ba_top5 = balanced_accuracy_score(result["label"], top5_pred_adj)

my_metrics = {
    "total_evaluated":        n,
    "top1_accuracy":          round(top1_acc, 6),
    "top1_balanced_accuracy": round(ba_top1,  6),
    "top5_accuracy":          round(top5_acc, 6),
    "top5_balanced_accuracy": round(ba_top5,  6),
}
my_metrics

/user/h.siddiqui/u27673/.conda/envs/vbbf/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/user/h.siddiqui/u27673/.conda/envs/vbbf/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


{'total_evaluated': 150,
 'top1_accuracy': np.float64(0.486667),
 'top1_balanced_accuracy': np.float64(0.486667),
 'top5_accuracy': np.float64(0.9),
 'top5_balanced_accuracy': np.float64(0.9)}

- cam0 = birds eye
- cam1 = POV
- cam2 = side view

In [8]:
result.groupby("cam")[["correct_top1", "correct_top5"]].mean().rename(
        columns={"correct_top1": "top1_acc", "correct_top5": "top5_acc"}
)

,top1_acc,top5_acc
cam,,
cam0,0.40,0.86
cam1,0.62,0.98
cam2,0.44,0.86


In [9]:
result.groupby("template")[["correct_top1", "correct_top5"]].mean().rename(
        columns={"correct_top1": "top1_acc", "correct_top5": "top5_acc"}
).sort_values("template")

,top1_acc,top5_acc
template,,
Attaching [something] to [something],0.40,0.87
Moving [something] up,0.07,1.00
Piling [something] up,0.00,1.00
Plugging [something] into [something],1.00,1.00
Pushing [something] with [something],1.00,1.00
Putting [something] next to [something],0.40,0.93
Putting [something] on a surface,0.47,1.00
Putting [something] onto [something],0.27,0.93
Showing a photo of [something] to the camera,0.27,0.27


In [10]:
result[result.cam=='cam1'].groupby("template")[["correct_top1", "correct_top5"]].mean().rename(
        columns={"correct_top1": "top1_acc", "correct_top5": "top5_acc"}
).sort_values("template")

,top1_acc,top5_acc
template,,
Attaching [something] to [something],0.4,1.0
Moving [something] up,0.2,1.0
Piling [something] up,0.0,1.0
Plugging [something] into [something],1.0,1.0
Pushing [something] with [something],1.0,1.0
Putting [something] next to [something],0.6,1.0
Putting [something] on a surface,0.6,1.0
Putting [something] onto [something],0.6,1.0
Showing a photo of [something] to the camera,0.8,0.8
